# KnowledgeForge AI - Programmatic API Tutorial 🚀

Welcome Google Gemini XPRIZE Hackathon Judges!

This notebook demonstrates how to interact with the core engine of **KnowledgeForge AI** completely programmatically, without using the Streamlit or PyWebView frontends.

This highlights the modularity, security, and raw power of our custom FAISS + Cross-Encoder RAG pipeline.

In [13]:
import sys
from pathlib import Path

# Add the project root to the path so we can import the modules
project_root = Path("..").resolve()
sys.path.append(str(project_root))

from rag.vector_store import ChromaVectorStore as FAISSVectorStore
from rag.embeddings import EmbeddingService
from rag.retrieval import HybridRetriever
from rag.reranker import CrossEncoderReranker
from rag.generator import RAGGenerator
from models.llm import DeepsetGemmaLLM

print("All core KnowledgeForge AI modules loaded successfully!")

All core KnowledgeForge AI modules loaded successfully!


### 1. Initialize the Core Engine
We instantiate our FAISS vector store, Hybrid Retriever (Dense + BM25), Cross-Encoder Reranker, and our Local Gemma LLM Synthesizer.

In [14]:
# 1. Load the extremely fast FAISS database
vector_store = FAISSVectorStore()

# 2. Load the embedding model (all-MiniLM-L6-v2 by default)
embedding_service = EmbeddingService()

# 3. Create the Hybrid Retriever (combines BM25 sparse search with FAISS dense search)
retriever = HybridRetriever(vector_store, embedding_service)

# 4. Load the Cross-Encoder for advanced contextual reranking
reranker = CrossEncoderReranker()

# 5. Boot up the local LLM Synthesizer (connects to local Ollama API)
llm = DeepsetGemmaLLM(model_name="gemma:2b", use_native_weights=False)

# 6. Assemble the final RAG Generator pipeline
generator = RAGGenerator(retriever, reranker, llm)

print("Engine initialized. Ready for queries.")

2026-07-20 16:23:16 | INFO     | rag.vector_store: Loaded FAISS index with 27 vectors.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-07-20 16:23:22 | INFO     | models.embedding: Local EmbeddingModel (all-MiniLM-L6-v2) initialized successfully.
2026-07-20 16:23:22 | INFO     | rag.retrieval: BM25 index built across 27 chunks.
Engine initialized. Ready for queries.


### 2. Run a Query against the Database
Let's ask the engine a question. It will retrieve the most relevant chunks, rerank them for extreme precision, and instruct Gemma to generate an answer purely based on the local context.

In [15]:
query = "What is KnowledgeForge AI?"
print(f"User Query: {query}\n")

answer, sources = generator.generate_response(query, top_k=5, use_reranker=True)

print("==================== AI RESPONSE ====================")
print(answer)
print("=====================================================\n")

print("--- Citations & Sources ---")
for i, source in enumerate(sources, 1):
    meta = source.get('metadata', {})
    print(f"Source {i}: {meta.get('source', 'Unknown')} (Page {meta.get('page', 1)})")
    
    conf = source.get('confidence_estimate', {}).get('score_percentage', 0) / 100.0
    if 'rerank_score' in source:
        conf = source['rerank_score']
    print(f"Confidence Score: {conf:.4f}")
    print("-" * 30)

User Query: What is KnowledgeForge AI?

2026-07-20 16:23:22 | INFO     | rag.generator: Retrieved 15 hybrid candidates for query.
2026-07-20 16:23:22 | INFO     | rag.reranker: Loading CrossEncoder reranker model: BAAI/bge-reranker-base...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-07-20 16:23:29 | INFO     | rag.reranker: CrossEncoder loaded successfully.
2026-07-20 16:23:37 | INFO     | rag.generator: Selected top 5 reranked chunks.
2026-07-20 16:23:37 | INFO     | models.llm: Executing local inference using `gemma:2b` (native_weights=False)...
2026-07-20 16:23:38 | INFO     | models.llm: llama-cpp is offline. Falling back to Ollama REST API (model: gemma:2b)...
==================== AI RESPONSE ====================
The context does not provide any information about KnowledgeForge AI, so I cannot answer this question from the provided context.

--- Citations & Sources ---
Source 1: JavaScript Essentials.pdf (Page 1)
Confidence Score: 0.0007
------------------------------
Source 2: JavaScript Essentials.pdf (Page 1)
Confidence Score: 0.0002
------------------------------
Source 3: JavaScript Essentials.pdf (Page 6)
Confidence Score: 0.0001
------------------------------
Source 4: JavaScript Essentials.pdf (Page 1)
Confidence Score: 0.0001
-------------------